# Azure ML Training Job: Registry Data → Prepare → Train

Compute Instance는 Notebook을 실행하고 Job을 제출하는 workstation입니다. 실제 Training Job은 팀의 `cpu-secure` Compute Cluster에서 실행합니다.

기존 `demo/data.py`, `demo/training.py`를 그대로 사용합니다. Registry의 합성 Data v1과 CPU Environment v1을 참조하며 기존 Endpoint는 변경하지 않습니다.

Studio Notebooks에서 이 파일을 열고 해당 팀의 `ci-kt-teamN`을 선택한 후 위에서부터 실행하세요. Python kernel을 사용합니다.

In [ ]:
%pip install --quiet --index-url https://pypi.org/simple "azure-ai-ml>=1.30,<2" "azure-identity>=1.25,<2" "mlflow==2.22.2" "azureml-mlflow>=1.60,<2"

## 1. 해당 팀 Workspace 연결

`config.json`은 팀별로 배치됩니다. Notebook의 User-assigned Managed Identity를 사용하며 secret·Storage key는 포함하지 않습니다.
아래 셀은 Workspace read API를 실제 호출합니다. `Authenticated Workspace` 출력은 인증·조회 성공 후에만 나타납니다. Studio의 Compute Authenticate는 별도의 사용자 로그인입니다.

In [ ]:
import json
import os
from pathlib import Path
import mlflow
from azure.ai.ml import MLClient, Input, Output, command, dsl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedIdentityConfiguration, Model
from azure.identity import DefaultAzureCredential

sample = Path.home() / 'cloudfiles/code/Users/kt-demo/kt-mlops-training'
if not (sample / 'config.json').is_file():
    sample = Path.cwd()
config = json.loads((sample / 'config.json').read_text(encoding='utf-8'))
# MLflow의 별도 credential chain에도 동일한 User-assigned Identity를 지정합니다.
os.environ['AZURE_CLIENT_ID'] = config['notebook_identity_client_id']
credential = DefaultAzureCredential(
    managed_identity_client_id=config['notebook_identity_client_id'],
    process_timeout=600,  # Local Azure CLI token acquisition can exceed the default 10 seconds.
)
ml = MLClient.from_config(credential=credential, path=str(sample / 'config.json'))
code = sample / 'src'
if not (code / 'demo/training.py').is_file():
    raise FileNotFoundError(f'Training source is missing: {code}')
workspace = ml.workspaces.get(ml.workspace_name)
print('Authenticated Workspace:', workspace.name)


## 2. Prepare / Train Component와 Pipeline 정의

동일한 Registry Data v1, 75/25 stratified holdout, seed 42를 사용합니다. 두 알고리즘의 Metrics는 각각의 Train Child Job에 기록됩니다.
각 단계는 해당 팀 Compute Cluster의 Managed Identity로 Data와 Outputs에 접근합니다.

In [ ]:
registry = 'azureml://registries/mlr-ktmlops-shared'
environment = f'{registry}/environments/kt-shared-cpu/versions/1'
data_asset = f'{registry}/data/telco-churn-shared/versions/1'
prepare = command(
    display_name='Prepare synthetic telco holdout',
    code=str(code), environment=environment,
    inputs={'data': Input(type=AssetTypes.URI_FILE)},
    outputs={'prepared': Output(type=AssetTypes.URI_FOLDER)},
    command='python -m demo.data --source ${{inputs.data}} --destination ${{outputs.prepared}}',
)
train = command(
    display_name='Train and log MLflow classifier',
    code=str(code), environment=environment,
    inputs={'data': Input(type=AssetTypes.URI_FOLDER), 'algorithm': Input(type='string')},
    outputs={'model': Output(type=AssetTypes.MLFLOW_MODEL)},
    command='python -m demo.training --source ${{inputs.data}} --destination ${{outputs.model}} '
            '--algorithm ${{inputs.algorithm}} --data-version 1',
)

@dsl.pipeline(default_compute="cpu-secure", display_name='Notebook telco training')
def training_pipeline(algorithm: str):
    prepared = prepare(data=Input(type=AssetTypes.URI_FILE, path=data_asset))
    prepared.identity = ManagedIdentityConfiguration()
    trained = train(data=prepared.outputs.prepared, algorithm=algorithm)
    trained.identity = ManagedIdentityConfiguration()
    return {'model': trained.outputs.model}


## 3. Training Jobs 제출

이 셀을 실행할 때마다 두 개의 새로운 Pipeline Job이 생성됩니다. `kt-notebook-training` Experiment에서 Parent Job과 Graph를 확인합니다.
`cpu-secure`는 max 1 node이므로 실행이 대기할 수 있습니다.

In [ ]:
jobs = []
for algorithm in ('logistic', 'forest'):
    pipeline = training_pipeline(algorithm)
    pipeline.display_name = f'notebook-{algorithm}-data-v1'
    pipeline.tags = {'algorithm': algorithm, 'data_version': '1', 'source': 'notebook'}
    pipeline.settings.force_rerun = True
    job = ml.jobs.create_or_update(pipeline, experiment_name='kt-notebook-training')
    jobs.append(job)
    print(algorithm, job.name, job.studio_url)


In [ ]:
for job in jobs:
    ml.jobs.stream(job.name)
    completed = ml.jobs.get(job.name)
    if completed.status != 'Completed':
        raise RuntimeError(f'{job.name}: {completed.status}')


## 4. Train Child Run의 Metrics 비교

Parent Pipeline의 Metrics가 아니라 실제 MLflow 기록이 있는 Train Child Run을 조회합니다.
각 Train Job의 Outputs + logs에서 confusion matrix, metrics JSON, MLflow Model도 확인하세요.

In [ ]:
mlflow.set_tracking_uri(ml.workspaces.get(ml.workspace_name).mlflow_tracking_uri)
tracking = mlflow.MlflowClient()
experiment = tracking.get_experiment_by_name('kt-notebook-training')
if experiment is None:
    raise RuntimeError('Training Experiment was not found')
results = []
for job in jobs:
    children = tracking.search_runs(
        [experiment.experiment_id], filter_string=f"tags.mlflow.parentRunId = '{job.name}'"
    )
    trained = [run for run in children if 'roc_auc' in run.data.metrics]
    if len(trained) != 1:
        raise RuntimeError(f'Expected one Train Run with Metrics: {job.name}')
    run = trained[0]
    result = {'parent_job': job.name, 'train_run': run.info.run_id,
              'algorithm': run.data.params['algorithm'], **run.data.metrics}
    results.append(result)
    print(json.dumps(result, indent=2))


## 5. Optional: Model 버전 등록

기본값은 비활성화입니다. 등록하려면 아래 flag를 변경하세요. 원본 샘플과 다른 Model 이름을 사용하며 기존 Endpoint는 변경하지 않습니다.
이는 자동 best-model selection이나 production promotion이 아닙니다.

In [ ]:
enable_model_registration = False
if enable_model_registration:
    for job in jobs:
        registered = ml.models.create_or_update(Model(
            name='telco-notebook-classifier', version=job.name,
            path=f'azureml://jobs/{job.name}/outputs/model', type=AssetTypes.MLFLOW_MODEL,
            tags={'source': 'notebook', 'data_version': '1'},
        ))
        print(registered.name, registered.version)
